<h1>Chapter 8 - Agentic RAG</h1>
<i>Building a multi-agent system using OpenAI SDK.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch08_agentic_rag/8.6_sales_negotiation_agent_openai_sdk/building_agents_with_openai_sdk.ipynb)

---

This notebook is for Chapter 8 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


## Building a multi-agent system using OpenAI SDK

## Install Required Packages

In [1]:
!pip install openai-agents==0.9.2 chromadb==1.5.0

### Load secrets

If you run this code in Google Colab, save your OpenAI API key in the secrets and access it by

In [2]:
import os
import sys
from dotenv import load_dotenv

# Check if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    try:
        from google.colab import userdata  # type: ignore

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except ModuleNotFoundError:
        pass
else:
    load_dotenv()

### Load sample files

This notebook uses sample Word and PDF files.

When running the notebook on Google Colab, uncomment the code below to download the `datasets` directory from the Github repo.

In [ ]:
import sys

# Check if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone --no-checkout https://github.com/polzerdo55862/RAG-with-Python-Cookbook.git
    %cd RAG-with-Python-Cookbook
    !git sparse-checkout init --cone
    !git sparse-checkout set datasets
    !git checkout
    !cp -r datasets /content/datasets

## Code

In [7]:
import os
import shutil
import chromadb
from chromadb.config import Settings

def fill_chroma_db():
    chroma_dir = "./chroma_email_history_db"
    # If the directory exists, remove it to overwrite
    if os.path.exists(chroma_dir):
        shutil.rmtree(chroma_dir)
    client = chromadb.Client(Settings(persist_directory=chroma_dir))
    collection = client.get_or_create_collection("email_history")

    with open(
        "../datasets/sample_data_negotiation_agents/sample_email_history.txt", "r", encoding="utf-8"
    ) as f:
        email_text = f.read()

    email_docs = [
        e.strip() for e in email_text.split("\n---\n") if e.strip()
    ]

    for idx, doc in enumerate(email_docs):
        collection.add(documents=[doc], ids=[f"email_{idx+1}"])
        print(f"Added email_{idx+1}")

fill_chroma_db()


FileNotFoundError: [Errno 2] No such file or directory: '../datasets/sample_data_negotiation_agents/sample_email_history.txt'

In [ ]:
import chromadb
from chromadb.config import Settings
from agents import function_tool

@function_tool
def query_database(query_text, n_results=3):
    """
    Query emails using semantic search from a local ChromaDB collection.
    """

    # Set up ChromaDB client and collection
    chroma_dir = "./chroma_email_history_db"
    client = chromadb.Client(Settings(persist_directory=chroma_dir))
    collection = client.get_or_create_collection("email_history")

    # Perform the query
    results = collection.query(
        query_texts=[query_text],
        n_results=n_results,
        include=["documents"],
    )
    return {
        "query": query_text,
        "documents": (
            results["documents"][0] if results["documents"] else []
        ),
    }

In [ ]:
from agents import Agent, Runner
from agents import trace

# Create a negotiation agent with email search capability
salesman_agent = Agent(
    name="Negotiation Agent",
    instructions="""You are a skilled negotiation agent
    representing a salesperson.

    Your role:
    - Analyze the current email conversation history
    - Search for relevant information using the query_database tool
    - Generate informed responses with strategic counter-offers

    Process:
    1. Use query_database to find relevant context from email history
    2. Analyze customer concerns and pricing constraints
    3. Craft a professional response with a competitive counter-offer
    4. Maintain a collaborative tone while protecting profit margins
    """,
    tools=[query_database],
    model="gpt-5"
)

# Helper function to run the negotiation agent
async def negotiate_with_customer(email_history):
    with trace("Negotiation Response"):
        result = await Runner.run(
            salesman_agent,
            f"Email conversation history:\n{email_history}"
        )
    return result.final_output

In [ ]:
from agents import Agent, Runner

async def run_negotiation_process():
    """
    Orchestrates a negotiation between customer and salesperson agents
    using a moderator agent to facilitate the conversation.
    """
    # Load the email history from file
    with open(
        "../datasets/sample_data_negotiation_agents/sample_email_history.txt", "r", encoding="utf-8"
    ) as f:
        email_history = f.read()

    # Define the customer agent
    customer_agent = Agent(
        name="Customer",
        instructions="""Negotiate for the best laptop deal.
                       Be polite but persistent.
                       Respond to the email chain below.""",
        model="gpt-5-mini",
    )

    # Convert agents to tools for the moderator
    customer_agent_tool = customer_agent.as_tool(
        tool_name="customer",
        tool_description=(
            "Tool that represents the customer in negotiations."
        )
    )

    salesman_agent_tool = salesman_agent.as_tool(
        tool_name="salesman",
        tool_description=(
            "Tool that represents the salesperson in negotiations."
        )
    )

    # Define the moderator agent with clear instructions
    moderator_instructions = """
    You are a moderator facilitating negotiation between
    customer and salesperson.

    Process:
    1. Receive email history with the last message from customer
    2. Use salesman_agent_tool to generate salesperson response
    3. Append response to email history
    4. Use customer_agent_tool to generate customer response
    5. Continue alternating until agreement or breakdown

    Rules:
    - Aim for mutually beneficial agreement
    """

    moderator_agent = Agent(
        name="Moderator",
        instructions=moderator_instructions,
        model="gpt-5-mini",
        tools=[customer_agent_tool, salesman_agent_tool]
    )

    # Execute the negotiation with tracing
    with trace("Negotiation Process"):
        responses = await Runner.run(
            moderator_agent,
            f"Begin negotiation with this email history: {email_history}"
        )

    return responses


In [ ]:
import os
import shutil
import chromadb
from chromadb.config import Settings

def fill_chroma_db():
    chroma_dir = "./chroma_email_history_db"
    # If the directory exists, remove it to overwrite
    if os.path.exists(chroma_dir):
        shutil.rmtree(chroma_dir)
    client = chromadb.Client(Settings(persist_directory=chroma_dir))
    collection = client.get_or_create_collection("email_history")

    with open("../datasets/sample_data_negotiation_agents/sample_email_history.txt", "r", encoding="utf-8") as f:
        email_text = f.read()

    email_docs = [e.strip() for e in email_text.split("\n---\n") if e.strip()]

    for idx, doc in enumerate(email_docs):
        collection.add(documents=[doc], ids=[f"email_{idx+1}"])
        print(f"Added email_{idx+1}")

fill_chroma_db()

Added email_1


In [ ]:
# import helper_functions_agents_sdk
# from importlib import reload
# reload(helper_functions_agents_sdk)

# helper_functions_agents_sdk.query_emails("I give you 500 bucks")

In [ ]:
import chromadb
from chromadb.config import Settings

@function_tool
def query_database(query_text: str, n_results: int = 3) -> dict:
    """
    Query emails using semantic search from a local ChromaDB collection.
    """

    # Set up ChromaDB client and collection
    chroma_dir = "./chroma_email_history_db"
    client = chromadb.Client(Settings(persist_directory=chroma_dir))
    collection = client.get_or_create_collection("email_history")

    # Perform the query
    results = collection.query(
        query_texts=[query_text],
        n_results=n_results,
        include=["documents"],
    )
    return {
        "query": query_text,
        "documents": results["documents"][0] if results["documents"] else [],
    }

In [ ]:
from agents import Agent, Runner
from agents import trace

# Create a negotiation agent with email search capability
salesman_agent = Agent(
    name="Negotiation Agent",
    instructions="""You are a skilled negotiation agent representing a salesperson.

    Your role:
    - Analyze the current email conversation history
    - Search for relevant information using the query_database tool
    - Generate informed responses with strategic counter-offers

    Process:
    1. Use query_database to find relevant context from email history
    2. Analyze customer concerns and pricing constraints
    3. Craft a professional response with a competitive counter-offer
    4. Maintain a collaborative tone while protecting profit margins
    """,
    tools=[query_database],
    model="gpt-5"
)

# Helper function to run the negotiation agent
async def negotiate_with_customer(email_history: str) -> str:
    with trace("Negotiation Response"):
        result = await Runner.run(
            salesman_agent,
            f"Email conversation history:\n{email_history}"
        )
    return result.final_output

In [ ]:
import asyncio
from agents import trace

async def demonstrate_negotiation():
    with open("../datasets/sample_data_negotiation_agents/sample_email_history.txt",
              "r", encoding="utf-8") as f:
        email_history = f.read()

    # Generate negotiation response using the agent
    with trace("Automated Customer Response"):
        response = await negotiate_with_customer(email_history)

    print(response)
    return response

await demonstrate_negotiation()

I tried to pull the email thread from our database but only got “404: Not Found.” Since I don’t have the specifics of the customer’s ask, here’s a ready-to-send reply you can adapt. It anchors value, offers conditional concessions, and protects margin.

Subject: Aligning on scope and pricing

Hi [Name],

Thanks for the candid feedback. I want to make this work within your budget while keeping the outcomes we discussed. Here are a few paths we can take—pick the one that best fits, and I’ll update the order:

- Option A — Keep full scope, annual: If we do annual billing, I can extend a [~10–12%] reduction and include white-glove onboarding and priority support. This keeps the full package intact and gets you live quickly.

- Option B — Multi-year value: With a 24–36 month term, I can reach [~15–18%] and lock pricing for the term. I can also add flexible payment terms (e.g., quarterly) and two admin training credits. In exchange, we’d look for a reference call and permission to use logo/c

'I tried to pull the email thread from our database but only got “404: Not Found.” Since I don’t have the specifics of the customer’s ask, here’s a ready-to-send reply you can adapt. It anchors value, offers conditional concessions, and protects margin.\n\nSubject: Aligning on scope and pricing\n\nHi [Name],\n\nThanks for the candid feedback. I want to make this work within your budget while keeping the outcomes we discussed. Here are a few paths we can take—pick the one that best fits, and I’ll update the order:\n\n- Option A — Keep full scope, annual: If we do annual billing, I can extend a [~10–12%] reduction and include white-glove onboarding and priority support. This keeps the full package intact and gets you live quickly.\n\n- Option B — Multi-year value: With a 24–36 month term, I can reach [~15–18%] and lock pricing for the term. I can also add flexible payment terms (e.g., quarterly) and two admin training credits. In exchange, we’d look for a reference call and permission to

In [ ]:
with open("../datasets/sample_data_negotiation_agents/sample_email_history.txt", "r", encoding="utf-8") as f:
    email_history = f.read()

email_history

'404: Not Found'

In [ ]:
from agents import Agent, Runner

async def run_negotiation_process():
    """
    Orchestrates a negotiation between customer and salesperson agents
    using a moderator agent to facilitate the conversation.
    """
    # Load the email history from file
    with open("../datasets/sample_data_negotiation_agents/sample_email_history.txt", "r", encoding="utf-8") as f:
        email_history = f.read()

    # Define the customer agent
    customer_agent = Agent(
        name="Customer",
        instructions="""Negotiate for the best laptop deal.
                       Be polite but persistent.
                       Respond to the email chain below.""",
        model="gpt-5-mini",
    )

    # Convert agents to tools for the moderator
    customer_agent_tool = customer_agent.as_tool(
        tool_name="customer",
        tool_description="Tool that represents the customer in negotiations."
    )

    salesman_agent_tool = salesman_agent.as_tool(
        tool_name="salesman",
        tool_description="Tool that represents the salesperson in negotiations."
    )

    # Define the moderator agent with clear instructions
    moderator_instructions = """
    You are a moderator facilitating negotiation between customer and salesperson.

    Process:
    1. Receive email history with the last message from customer
    2. Use salesman_agent_tool to generate salesperson response
    3. Append response to email history
    4. Use customer_agent_tool to generate customer response
    5. Continue alternating until agreement or breakdown

    Rules:
    - Aim for mutually beneficial agreement
    """

    moderator_agent = Agent(
        name="Moderator",
        instructions=moderator_instructions,
        model="gpt-5-mini",
        tools=[customer_agent_tool, salesman_agent_tool]
    )

    # Execute the negotiation with tracing
    with trace("Negotiation Process"):
        responses = await Runner.run(
            moderator_agent,
            f"Begin negotiation with this email history: {email_history}"
        )

    return responses

In [ ]:
responses = await run_negotiation_process()

In [ ]:
from agents import Agent, Runner

async def run_negotiation_process_with_handoffs():
    """
    Direct negotiation between customer and salesperson agents
    using handoffs to alternate between them.
    """
    # Load the email history from file
    with open("../datasets/sample_data_negotiation_agents/sample_email_history.txt", "r", encoding="utf-8") as f:
        email_history = f.read()

    # Define the salesperson agent
    salesman_agent = Agent(
        name="Salesperson",
        instructions="""You are a laptop salesperson negotiating a deal.
                       Make reasonable offers and try to close the sale.
                       If you reach agreement, clearly state 'DEAL AGREED'.
                       If negotiation isn't working after 3 rounds, politely end it.""",
        model="gpt-5-mini",
        handoff_description="Hand off to customer for their response to your offer."
    )

    # Define the customer agent with handoff capabilities
    customer_agent = Agent(
        name="Customer",
        instructions="""Negotiate for the best laptop deal.
                       Be polite but persistent.
                       Make counteroffers or accept good deals.
                       If you accept a deal, clearly state 'DEAL ACCEPTED'.
                       If you can't reach agreement after 3 rounds, politely walk away.""",
        model="gpt-5-mini",
        handoff_description="Hand off to salesperson for their response to your counteroffer."
    )

    # Convert each agent to a tool for the other
    customer_tool = customer_agent.as_tool(
        tool_name="handoff_to_customer",
        tool_description="Hand the conversation to the customer agent."
    )

    salesman_tool = salesman_agent.as_tool(
        tool_name="handoff_to_salesperson",
        tool_description="Hand the conversation to the salesperson agent."
    )

    # Give each agent the ability to hand off to the other
    customer_agent.tools = [salesman_tool]
    salesman_agent.tools = [customer_tool]

    # Start with the salesperson responding to the email history
    with trace("Direct Negotiation Process"):
        responses = await Runner.run(
            salesman_agent,
            f"Respond to this email history and begin negotiation: {email_history}"
        )

    return responses

In [ ]:
responses = await run_negotiation_process_with_handoffs()

In [ ]:
responses

RunResult(input='Respond to this email history and begin negotiation: 404: Not Found', new_items=[ReasoningItem(agent=Agent(name='Salesperson', handoff_description='Hand off to customer for their response to your offer.', tools=[FunctionTool(name='handoff_to_customer', description='Hand the conversation to the customer agent.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function Agent.as_tool.<locals>._run_agent_tool at 0x7d3067d1e8e0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None)], mcp_servers=[], mcp_config={}, instructions="You are a laptop salesperson negotiating a deal.\n                       Make reas